# MHRAG: GPU experiments on Kaggle (resumable)

Kaggle gives about 30 free GPU hours a week, in sessions of at most 12 hours. The full benchmark needs several sessions,
so this notebook is built to be run again and again with **Save Version → Save & Run All (Commit)**:

* **Steps.** Each experiment is a step (cells 7–11). When a step finishes, a marker is written to
  `mhrag_results/_checkpoints/`, and `/kaggle/working/mhrag_results.zip` is updated after every step.
* **Finished steps are skipped.** A later run skips every step with a marker. The marker also records a fingerprint of
  the step's config and eval data; if you change those, the step runs again.
* **Unfinished steps continue where they stopped.** Every generated answer, every retrieval run and every LLM-judge
  verdict is saved as soon as it exists. Answers that failed with an error are retried. A step interrupted by the time limit continues from there next session.
  Generation also skips models whose answers are all saved, and deletes each model's weights once it is done, so the
  disk does not fill up.
* **The session always saves its output.** `SESSION_HOURS` (cell 1) is a budget below Kaggle's 12-hour limit. When it
  runs out, the running step is stopped cleanly, no new step starts, and the notebook zips and finishes normally.

**Continuing in a new session.** In the editor: *Add Input → Your Work → Notebooks →* this notebook (its latest version),
then *Save Version → Save & Run All*. Cell 5 restores everything from that output. This also works with the output of
the older version of this notebook.

**One-time settings (right-hand panel):** *Accelerator* GPU T4 x2 (or P100); *Internet* on (needs a phone-verified
account); optional *Secrets* `HF_TOKEN` (gated Llama/Gemma models; accept their licences on Hugging Face first) and
`GROQ_API_KEY` (the LLM judge in step 10 and the API latency row), each ticked for this notebook.

Every number the scripts write is a real measurement. Nothing is filled in by hand.

In [ ]:
# 1) Settings: which steps run, and the session time budget
import time
T0 = time.time()                  # the session clock starts here

SESSION_HOURS = 11.25             # Kaggle stops GPU sessions at 12 h; stop, zip and finish before that
BRANCH = "main"

RUN = {
    "classifier": False,          # already computed on the laptop (results/risk_classifier.json); True = recompute
    "retrieval_main": False,      # already computed on the laptop (results/retrieval_main.json)
    "retrieval_chunks": True,     # chunk-size ablation: 128/256/512 tokens x 5 embedders
    "latency": True,              # CUDA latency: bf16, 4-bit NF4, API model
    "generation_local": True,     # deployed Qwen2.5-1.5B GGUF + the v1 models (GPT-2, BART)
    "generation_gpu": False,      # the 19-model benchmark (several sessions): set True to run it
}
JUDGE_MODEL = None                # LLM judge from configs/models.yaml; None = the experiment's (gpt-oss-120b-groq).
                                  # If cell 5b fails, its message lists the models your Groq key can use
GEN_GPU_MODELS = None             # None = every model, continuing where the last session stopped;
                                  # or a list, e.g. ["qwen2.5-7b-instruct", "llama-3.1-8b-instruct"]
FINISH_WITHOUT = []               # benchmark models to give up on (e.g. gated access never granted): scoring
                                  # then runs without them, and they are reported as TODO(run)

In [ ]:
# 2) GPU and internet check
!nvidia-smi || echo "No GPU: set Accelerator to GPU T4 x2 in the notebook settings"
!curl -sI https://huggingface.co | head -1 || echo "No internet: turn Internet on in the notebook settings"

In [ ]:
# 3) Clone the repository
REPO_URL = "https://github.com/HellDragger/MentalHealthRAG-Chatbot.git"   # change if you forked it
REPO = "/kaggle/working/mhrag"
!rm -rf {REPO} && git clone -b {BRANCH} --depth 1 {REPO_URL} {REPO}
%cd {REPO}
!git log --oneline -1

In [ ]:
# 4) Install (GPU extras + evaluation tools + 4-bit loading). Kaggle images already ship torch with CUDA.
!pip -q install -e ".[gpu,eval,dev]" bitsandbytes

In [ ]:
# 5) Secrets, results directory, resume from earlier sessions, and the step runner
import datetime, glob, hashlib, json, os, shutil, signal, subprocess, sys, threading, time, zipfile

try:
    from kaggle_secrets import UserSecretsClient
    sc = UserSecretsClient()
    for k in ("HF_TOKEN", "GROQ_API_KEY", "OPENROUTER_API_KEY"):
        try:
            os.environ[k] = sc.get_secret(k)
        except Exception:
            pass
except ImportError:
    print("not running on Kaggle")

PY = sys.executable
RESULTS = "/kaggle/working/mhrag_results"
ZIP = "/kaggle/working/mhrag_results.zip"
MARKERS = os.path.join(RESULTS, "_checkpoints")
os.makedirs(MARKERS, exist_ok=True)

# a) Start from the committed laptop results, so merged files (latency.json) and the paper numbers include them.
for src in glob.glob(os.path.join(REPO, "results", "*")):
    dst = os.path.join(RESULTS, os.path.basename(src))
    if not os.path.exists(dst):
        (shutil.copytree if os.path.isdir(src) else shutil.copy2)(src, dst)

# b) Restore earlier sessions: the output of a previous version of this notebook, attached with Add Input.
def _size(d):
    return sum(os.path.getsize(p) for p in glob.glob(os.path.join(d, "**"), recursive=True) if os.path.isfile(p))

prev = [d for d in glob.glob("/kaggle/input/**/mhrag_results", recursive=True) if os.path.isdir(d)]
if not prev:  # only the zip is there
    for i, z in enumerate(glob.glob("/kaggle/input/**/mhrag_results.zip", recursive=True)):
        zipfile.ZipFile(z).extractall(f"/tmp/prev_{i}")
        prev.append(f"/tmp/prev_{i}")
for d in sorted(prev, key=_size):  # the largest (most complete) output is copied last and wins
    shutil.copytree(d, RESULTS, dirs_exist_ok=True)
    print("resumed from", d)
if not prev:
    print("No earlier output attached: starting fresh.")

os.environ.update({
    "MHRAG_PATHS__RESULTS": RESULTS,
    "MHRAG_LOAD_IN_4BIT": "1",          # NF4 for 7-12B models on a 16 GB T4
    "MHRAG_FREE_MODEL_CACHE": "1",      # delete each model's weights once all its answers are saved (disk space)
    "TOKENIZERS_PARALLELISM": "false",
    "PYTHONUNBUFFERED": "1",
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",  # less fragmentation across many models in one process
})
if JUDGE_MODEL:
    os.environ["MHRAG_JUDGE_MODEL"] = JUDGE_MODEL

# Files that define each step; a marker only counts if they are unchanged.
EVAL_DATA = ["eval/data/heading_qa.jsonl", "eval/data/paraphrase_qa.jsonl", "eval/data/synth_qa.jsonl"]
GEN_DATA = ["eval/data/faq_gen.jsonl", "eval/data/oos_questions.jsonl", "eval/data/counsel_gen.jsonl"]
STEP_FILES = {
    "classifier": ["scripts/train_risk_classifier.py", "eval/data/safety_prompts.jsonl",
                   "eval/data/safety_prompts_heldout.jsonl"],
    "retrieval_main": ["configs/experiments/retrieval_main.yaml"] + EVAL_DATA,
    "retrieval_chunks": ["configs/experiments/retrieval_chunks.yaml"] + EVAL_DATA,
    "latency": ["scripts/bench_latency.py"],
    "generation_local": ["configs/experiments/generation_local.yaml"] + GEN_DATA,
    "generation_gpu": ["configs/experiments/generation_gpu.yaml"] + GEN_DATA,
}


def hours_left():
    return SESSION_HOURS - (time.time() - T0) / 3600


def _fingerprint(step):
    h = hashlib.sha1()
    for f in STEP_FILES.get(step, []):
        with open(os.path.join(REPO, f), "rb") as fh:
            h.update(fh.read())
    return h.hexdigest()[:12]


def is_done(step):
    path = os.path.join(MARKERS, step + ".json")
    if not os.path.exists(path):
        return False
    with open(path) as f:
        return json.load(f).get("fingerprint") == _fingerprint(step)


def mark_done(step, minutes, **extra):
    commit = subprocess.run(["git", "-C", REPO, "rev-parse", "--short", "HEAD"], capture_output=True, text=True)
    with open(os.path.join(MARKERS, step + ".json"), "w") as f:
        json.dump({"step": step, "finished": datetime.datetime.now().isoformat(timespec="seconds"),
                   "minutes": round(minutes, 1), "fingerprint": _fingerprint(step),
                   "commit": commit.stdout.strip(), **extra}, f, indent=1)


def save_checkpoint(label):
    shutil.make_archive(ZIP[:-4], "zip", RESULTS)
    print(f"[zip]  after {label}: {ZIP} ({os.path.getsize(ZIP) / 2**20:.1f} MB); {hours_left():.2f} h left")


def _stream(cmds, timeout_s, log):
    """Run one shell command, or a list of commands in parallel (e.g. one per GPU), with live output. When the
    budget runs out, every command and its child processes are stopped. Returns the exit codes (None = stopped)."""
    cmds = cmds if isinstance(cmds, list) else [cmds]
    procs, threads = [], []
    for i, cmd in enumerate(cmds):
        p = subprocess.Popen(cmd, shell=True, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                             bufsize=1, start_new_session=True)
        tag = f"[{i}] " if len(cmds) > 1 else ""

        def pump(p=p, tag=tag):
            for line in p.stdout:
                print(tag + line, end="", flush=True)
                log.write(tag + line)

        t = threading.Thread(target=pump, daemon=True)
        t.start()
        procs.append(p)
        threads.append(t)
    deadline = time.time() + max(1.0, timeout_s)
    rcs = []
    for p in procs:
        try:
            rcs.append(p.wait(timeout=max(1.0, deadline - time.time())))
        except subprocess.TimeoutExpired:
            rcs.append(None)
    for p, rc in zip(procs, rcs):
        if rc is None:
            os.killpg(p.pid, signal.SIGTERM)
            try:
                p.wait(timeout=60)
            except subprocess.TimeoutExpired:
                os.killpg(p.pid, signal.SIGKILL)
                p.wait()
    for t in threads:
        t.join(timeout=30)
    return rcs


def run_step(step, commands, min_hours=0.25, complete=True):
    """Run a step unless it finished in an earlier session, then zip. `complete=False`: the commands cover only
    part of the step (a model subset), so no marker is written."""
    if not RUN.get(step, True):
        print(f"[off]  {step}: switched off in cell 1")
        return False
    if is_done(step):
        print(f"[skip] {step}: finished in an earlier session")
        return True
    if hours_left() < min_hours:
        print(f"[wait] {step}: only {hours_left():.2f} h left in this session; it runs next session")
        return False
    print(f"[run]  {step} ({hours_left():.2f} h left)", flush=True)
    t0, ok = time.time(), True
    os.makedirs(os.path.join(RESULTS, "logs"), exist_ok=True)
    with open(os.path.join(RESULTS, "logs", f"kaggle_{step}.log"), "a") as log:
        log.write(f"\n===== session started {datetime.datetime.now():%Y-%m-%d %H:%M}\n")
        for cmd in commands:
            for i, c in enumerate(cmd if isinstance(cmd, list) else [cmd]):
                label = f"$ [{i}] {c}" if isinstance(cmd, list) else f"$ {c}"
                print(label, flush=True)
                log.write(label + "\n")
            rcs = _stream(cmd, hours_left() * 3600, log)
            rc = None if None in rcs else next((r for r in rcs if r == 75), max(rcs))
            if rc is None:
                print(f"[stop] {step}: session budget reached. Saved progress continues next session.")
                ok = False
                break
            if rc == 75:
                print(f"[wait] {step}: API quota used up for today. Saved progress continues next session.")
                ok = False
                break
            if rc != 0:
                print(f"[fail] {step}: exit code {rc} (see logs/kaggle_{step}.log); it is retried next session")
                ok = False
                break
    if ok and complete:
        mark_done(step, (time.time() - t0) / 60)
        print(f"[done] {step} in {(time.time() - t0) / 60:.0f} min")
    save_checkpoint(step)
    return ok


# c) Output of the older notebook (before step markers existed): adopt experiments that finished there.
def _same_config(path, step):
    """The result was produced with the experiment config as it is now (e.g. not before a judge change)."""
    import yaml
    with open(path) as f:
        used = json.load(f).get("config") or {}
    with open(os.path.join(REPO, "configs", "experiments", f"{step}.yaml")) as f:
        now = yaml.safe_load(f)
    return all(used.get(k) == v for k, v in now.items())


def _finished(path, step):
    with open(path) as f:
        res = json.load(f)
    return (_same_config(path, step) and all(m.get("status") == "ok" for m in res.get("models", []))
            and res.get("judge_status", "complete") != "incomplete")


for step, out in [("retrieval_chunks", "retrieval_chunks.json"), ("generation_local", "generation_local.json"),
                  ("generation_gpu", "generation_gpu.json")]:
    path = os.path.join(RESULTS, out)
    if not os.path.exists(os.path.join(MARKERS, step + ".json")) and os.path.exists(path) and _finished(path, step):
        mark_done(step, 0, adopted_from_existing_results=True)
        print(f"[skip] {step}: finished results with the current config are already there")

print({k: bool(os.environ.get(k)) for k in ("HF_TOKEN", "GROQ_API_KEY")}, f"{hours_left():.2f} h left")
for step in STEP_FILES:
    print(f"  {step:18s} {'done' if is_done(step) else ('pending' if RUN.get(step) else 'off')}")

In [ ]:
# 5b) API check: the LLM judge and the API latency row need GROQ_API_KEY. A problem shows up here, not hours later.
if os.environ.get("GROQ_API_KEY"):
    CHECK = """
import os
from mhrag.llm.base import GenerationParams
from mhrag.llm.registry import load_catalog, make_backend
b = make_backend(load_catalog()[os.environ.get("MHRAG_JUDGE_MODEL", "gpt-oss-120b-groq")])
print("Judge", b.spec.key, "OK:", b.generate([{"role": "user", "content": "Reply with OK"}], GenerationParams(max_new_tokens=256, temperature=0)))
"""
    r = subprocess.run([PY, "-c", CHECK], cwd=REPO, capture_output=True, text=True)
    if r.returncode == 0:
        print(r.stdout.strip())
    else:
        print("Groq check FAILED: the LLM judge will be skipped; other metrics are unaffected.")
        print(r.stderr.strip()[-600:])
else:
    print("No GROQ_API_KEY: the LLM judge and the API latency row are skipped")

In [ ]:
# 6) Offline test suite (sanity check; a failure here is printed but does not stop the run)
!{PY} -m pytest -q -p no:cacheprovider | tail -3

In [ ]:
# 7) Risk classifiers (DistilRoBERTa fine-tunes in a few minutes on a T4) + safety-gate evaluation
# With access to the gated MentalRoBERTa, add: --transformer-model mental/mental-roberta-base
run_step("classifier", [
    f"{PY} -m scripts.train_risk_classifier",
    f"{PY} -m scripts.eval_safety --tag v2_devset",
    f"{PY} -m scripts.eval_safety --data eval/data/safety_prompts_heldout.jsonl --tag v2_heldout",
])

In [ ]:
# 8) Retrieval experiments. Indexes are built as needed (idempotent). The eval sets, including SynthQA (189 questions
#    generated with Qwen2.5-1.5B), are the committed ones in eval/data/, so every run uses identical queries.
run_step("retrieval_main", [f"{PY} -m scripts.run_eval --config configs/experiments/retrieval_main.yaml"])
run_step("retrieval_chunks", [f"{PY} -m scripts.run_eval --config configs/experiments/retrieval_chunks.yaml"])

In [ ]:
# 9) Latency on CUDA: bf16, 4-bit NF4 and the API model (merged into results/latency.json)
# One GPU only (CUDA_VISIBLE_DEVICES=0): with two visible, device_map="auto" splits a model across both T4s and
# the timings would not be single-T4 numbers.
ONE_GPU = "CUDA_VISIBLE_DEVICES=0"
run_step("latency", [
    f"{PY} -m scripts.build_index",   # the benchmark serves from the default index (idempotent)
    f"{ONE_GPU} MHRAG_LOAD_IN_4BIT=0 {PY} -m scripts.bench_latency --config v2_hf_cuda",
    f"{ONE_GPU} {PY} -m scripts.bench_latency --config v2_hf_cuda_4bit",
    f"{ONE_GPU} {PY} -m scripts.bench_latency --config v2_api || echo 'no GROQ_API_KEY: API latency skipped'",
])

### 10) Local-model generation (the laptop experiment, moved to Kaggle)
`configs/experiments/generation_local.yaml`: the deployed CPU model **Qwen2.5-1.5B-Instruct (GGUF Q4_K_M, llama.cpp)**
plus the original v1 models **GPT-2** and **BART-large-CNN**, each run without retrieval, with v1-style naive RAG and
with the full pipeline, on FAQ-Gen, the out-of-scope set and 50 Counsel-Gen questions.

The LLM judge (gpt-oss-120b on Groq) scores a fixed sample of 20 questions per dataset. Groq's free plan allows about
200k tokens a day for it, so the judge usually needs a few sessions: until the sample is complete this step reports
"judge incomplete" and runs again next session, reusing every verdict so far (answers are not generated again).

llama.cpp needs `llama-cpp-python`. The cell tries a prebuilt CUDA wheel first and otherwise compiles it with CUDA
(10–20 minutes). It is only installed while this step is still pending.

In [ ]:
def local_answers_done():  # only the LLM judge is left: llama.cpp is not needed
    import yaml
    with open(os.path.join(REPO, "configs/experiments/generation_local.yaml")) as f:
        models = yaml.safe_load(f)["models"]
    return all(os.path.exists(os.path.join(RESULTS, "generation", "local", f"{m}__status.json")) for m in models)

if RUN["generation_local"] and not is_done("generation_local") and not local_answers_done():
    def have_llama():
        return subprocess.run([PY, "-c", "import llama_cpp"], capture_output=True).returncode == 0
    if not have_llama():
        !pip -q install llama-cpp-python --prefer-binary --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124
    if not have_llama():
        !CMAKE_ARGS="-DGGML_CUDA=on" pip -q install llama-cpp-python --no-binary llama-cpp-python
    print("llama_cpp available:", have_llama())

run_step("generation_local", [
    f"MHRAG_LOAD_IN_4BIT=0 MHRAG_LLAMACPP_GPU_LAYERS=-1 {PY} -m scripts.run_eval "
    "--config configs/experiments/generation_local.yaml",
])

### 11) Generation benchmark (the long step)
19 open models × 3 settings × 248 questions. On a T4, 7–9B models in 4-bit take roughly 1–2 hours each, so this
takes several sessions. Set `RUN["generation_gpu"] = True` in cell 1. Each session continues where the previous one
stopped. Scoring (NLI, BERTScore) runs once all models have their answers; no LLM judge or API models here (free API quotas).

Models are generated two at a time, one per GPU. A model that cannot load yet (for example while gated access on
Hugging Face is pending) is recorded and tried again next session; the others carry on. Scoring runs, and the step is
marked finished, only once every model has its answers. To finish without a model, add it to `FINISH_WITHOUT` in cell 1.

In [ ]:
import yaml

GEN = f"{PY} -m scripts.run_eval --config configs/experiments/generation_gpu.yaml"
with open(os.path.join(REPO, "configs/experiments/generation_gpu.yaml")) as f:
    all_models = yaml.safe_load(f)["models"]
with open(os.path.join(REPO, "configs/models.yaml")) as f:
    params_b = {k: v.get("params_b") or 1 for k, v in yaml.safe_load(f)["models"].items()}
models = GEN_GPU_MODELS or all_models


def answered(m):  # all of the model's answers are saved (written when the model finishes)
    return os.path.exists(os.path.join(RESULTS, "generation", "gpu", f"{m}__status.json"))


if not RUN["generation_gpu"]:
    print("[off]  generation_gpu: switched off in cell 1")
elif is_done("generation_gpu"):
    print("[skip] generation_gpu: finished in an earlier session")
else:
    # Phase 1, answers: every model without answers yet. A model that cannot load (e.g. gated access still pending)
    # fails fast and is tried again next session.
    todo = [m for m in models if not answered(m)]
    n_gpu = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.count("GPU ")
    if todo:
        # One model per GPU, two at a time: each model gets a whole T4 (as in the paper) and it takes about half
        # the time. Largest models first, each to the GPU with less work so far.
        n = 2 if n_gpu >= 2 and len(todo) > 1 else 1
        groups, load = [[] for _ in range(n)], [0.0] * n
        for m in sorted(todo, key=lambda k: -params_b.get(k, 1)):
            g = load.index(min(load))
            groups[g].append(m)
            load[g] += params_b.get(m, 1)
        for i, g in enumerate(groups):
            print(f"GPU {i}:", g)
        generate = [f"CUDA_VISIBLE_DEVICES={i} {GEN} --generate-only --models {' '.join(g)}" for i, g in enumerate(groups)]
        run_step("generation_gpu", [generate if n > 1 else generate[0]], min_hours=0.5, complete=False)

    # Phase 2, scoring (NLI, BERTScore) and the finished marker: only once every model has its answers, so a model
    # whose access arrives later is still included.
    waiting = [m for m in models if not answered(m) and m not in FINISH_WITHOUT]
    if waiting:
        print(f"[wait] generation_gpu: scored once every model has answers; still without: {waiting}. Gated "
              "models: accept the licence on Hugging Face. To finish without a model, add it to FINISH_WITHOUT "
              "in cell 1 (it is then reported as TODO(run)).")
    else:
        score = f"CUDA_VISIBLE_DEVICES=0 {GEN}" + (" --models " + " ".join(GEN_GPU_MODELS) if GEN_GPU_MODELS else "")
        run_step("generation_gpu", [score], min_hours=0.25, complete=not GEN_GPU_MODELS)

In [ ]:
# 12) Human-evaluation sheets (blinded, randomised), once the generation benchmark is complete
if is_done("generation_gpu"):
    !{PY} -m eval.human_eval.make_sheets --exp gpu --raters 3 --n 60
else:
    print("generation_gpu not finished yet: sheets are made in a later session")

In [ ]:
# 13) Tables and paper numbers from everything finished so far, then the final zip
for exp in ("retrieval_main", "retrieval_chunks", "generation_local", "generation_gpu"):
    if os.path.exists(os.path.join(RESULTS, f"{exp}.json")):
        !{PY} -m scripts.run_eval --tables-only --config configs/experiments/{exp}.yaml
!{PY} -m scripts.paper_numbers
for sub in ("paper/tables", "paper/numbers.tex", "eval/human_eval/out"):
    src = os.path.join(REPO, sub)
    dst = os.path.join(RESULTS, "_repo_" + sub.replace("/", "_"))
    if os.path.isdir(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
    elif os.path.exists(src):
        shutil.copy(src, dst)
save_checkpoint("final")

print("\nStatus:")
for step in STEP_FILES:
    print(f"  {step:18s} {'done' if is_done(step) else ('pending' if RUN.get(step) else 'off')}")
print("\nNext session: Add Input -> this notebook's output, then Save Version -> Save & Run All.")
print("Locally: download mhrag_results.zip, then run: python -m scripts.import_results ~/Downloads/mhrag_results.zip")